# Assignment 34 -- Text Summarization using Langchain


# PART 1---Basic Text Summarization using PromptTemplate
## 1. Load the document using Langchain document loaders or manual loading.
## 2. Print:
## . Total character
## . Sample content preview


In [1]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "medical_health_data_document.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text = "\n".join(doc.page_content for doc in documents)

## print total charecters
print("Total characters:", len(text))

## sample content preview
print(text[:1000])


C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_15728\3903149179.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total characters: 7326
Medical Health Data: An Overview of Common
 Health Indicators
Educational document — 3–5 page reference material
This document provides general, educational information about commonly recorded medical health
indicators, preventive care, and basic interpretation of health data. It is intended for learning and does
not replace diagnosis or advice from a qualified healthcare professional.
1. Introduction to Medical Health Data
Medical health data is information collected about a person's physical condition, symptoms,
measurements, laboratory results, medical history, medications, and preventive-care activities.
Healthcare professionals use these data to understand health status, identify possible risks, monitor
changes over time, and support clinical decisions.
Common health measurements include body temperature, blood pressure, heart rate, respiratory rate,
oxygen saturation, body weight, height, and body mass index (BMI). Laboratory data may include blood
glucose,

# Task 2: Prompt-Based Summarization
## 1. Create a PromptTemplate for summarization:
## . System instruction(role as summarizer)
## . Placeholder for input text
## 2. Pass the text to an llm using langchain.
## 3. Generate and print the summary.

In [4]:

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Initialize model
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Load PDF document
loader = PyPDFLoader("medical_health_data_document.pdf")
data = loader.load()

# Create prompt template
prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert medical document summarizer.
Summarize the given medical document clearly and concisely.
Include important health information, main concepts,
and key points.

Medical Document:
{text}

Summary:
"""
)

# Create summarization chain
summary_chain = load_summarize_chain(
    model,
    prompt=prompt
)

# Generate summary
response = summary_chain.invoke({
    "input_documents": data
})

# Print summary
print("Medical Document Summary:")
print(response["output_text"])


Medical Document Summary:
**Medical Health Data – Key Points**

**1. What medical health data are**  
- Information on physical condition, symptoms, measurements, labs, history, meds, and preventive‑care activities.  
- Used by clinicians to assess status, spot risks, monitor trends, and guide decisions.

**2. Core Vital Signs (adult reference ranges – educational only)**  
| Sign | What it measures | Typical range |
|------|------------------|---------------|
| Body temperature | Core temperature | 36.1 – 37.2 °C |
| Resting heart rate | Beats per minute | 60 – 100 bpm |
| Respiratory rate | Breaths per minute | 12 – 20/min |
| Blood pressure | Systolic/diastolic arterial pressure | ≈120/80 mmHg (categories matter) |
| Oxygen saturation (SpO₂) | % of hemoglobin bound to O₂ | 95 – 100 % at sea level |

*Ranges vary with age, altitude, pregnancy, illness, meds, etc.*

**3. Blood Pressure & Cardiovascular Health**  
- Systolic = pressure during heart contraction; diastolic = pressure bet

# Task 3: Prompt Variations
## Create two prompt variations:
## 1. Short summary(5-6 lines) 
## 2. Bullet-point summary
## Compare outputs

In [19]:
from langchain_core.prompts import PromptTemplate

short_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert medical document summarizer.
Summarize the following text in 5-6 lines .
Focus on the main ideas and important information.
Use simple and clear language.
Avoid unnecessary details and repetition.
Keep the summary concise and meaningful.

Text: {text}""")

In [20]:
bullet_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template= """
Summarize the following text in bullet points.
Highlight the key ideas and important facts.
Use clear, simple, and readable language.
Keep each bullet point concise.
Avoid repetition and unnecessary details.

Text: {text}
""")

In [21]:

summary_chain = load_summarize_chain(
    model,
    prompt=short_summary_prompt
)


bullet_chain =load_summarize_chain(
    model,
    prompt=bullet_summary_prompt
)


In [24]:
short_response = summary_chain.invoke({
    "input_documents": data
})

bullet_response = bullet_chain.invoke({
    "input_documents": data
})



In [25]:

print("SHORT SUMMARY (5-6 LINES)")

print(short_response['output_text'])

print("\n")

print("BULLET-POINT SUMMARY")


print(bullet_response['output_text'])

SHORT SUMMARY (5-6 LINES)
- Medical health data includes vital signs (temperature, heart rate, breathing rate, blood pressure, oxygen saturation) and lab results such as glucose, cholesterol, kidney‑liver panels, and a complete blood count.  
- These measurements help clinicians assess current health, spot risks, and monitor changes over time, but interpretation always depends on the individual’s context.  
- Blood pressure, glucose, and lipid levels are key indicators of cardiovascular, diabetes, and heart‑disease risk; lifestyle changes and treatment can modify these risks.  
- CBC, kidney (creatinine, eGFR) and liver (ALT, AST, bilirubin) tests provide clues about anemia, infection, organ function, and other conditions.  
- Preventive‑care records—vaccinations, screenings, activity, diet, and substance use—support early detection and health‑maintenance planning.  
- Accurate, secure, and authorized handling of health data is essential, and any abnormal or concerning result should be

## Feature | Short Summary | Bullet-Point Summary 
## Format | Paragraph | Bullet points 
## Length | 5–6 lines | Multiple concise points 
## Readability | Good for quick reading | Very easy to scan 
## Detail | Limited | More key points can be included |


# PART 2 -- Stuff summarization chain

# Task 4: Why Stuff Chain is needed
## answer briefly:




## 1. What is a stuff summarization chain?
## A Stuff Chain combines all the document text into a single prompt and sends it to the LLM at once for summarization.

## 2. When is it suitable to use?
## It is suitable for small or medium-sized documents that fit within the model's context window.


## 3. Limitations of stuff chain.
## Its main limitation is that very large documents may exceed the context limit and consume more tokens, time, and API cost.

# Task 5: Implement stuff summarization chain
## 1. Use laod_summarize_chain() with:0 chain_type="stuff"
## 2. Pass the entire document as input
## 3. Generate summary

In [7]:

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.summarize import load_summarize_chain

# Initialize model
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Load PDF document
loader = PyPDFLoader("medical_health_data_document.pdf")
data = loader.load()

# Create summary prompt
prompt = PromptTemplate.from_template("""
You are an expert summarizer.

Summarize the following medical document
clearly and concisely in 5-6 lines.
Include only the main points.

Document:
{text}

Summary:
""")

# Create stuff summarization chain
summary_chain = load_summarize_chain(
    model,
    chain_type="stuff",
    prompt=prompt
)

# Generate summary
result = summary_chain.invoke({
    "input_documents": data
})

# Print summary
print(result["output_text"])



- Medical health data (vital signs, labs, history, preventive‑care) are tools for clinicians to assess status, risk, and trends, not substitutes for professional diagnosis.  
- Core vital‑sign ranges for adults: temperature ≈ 36.1‑37.2 °C, heart rate 60‑100 bpm, respiratory rate 12‑20/min, BP ≈ 120/80 mmHg, SpO₂ 95‑100 %.  
- Persistent high blood pressure raises cardiovascular/kidney risk; lifestyle changes and treatment are key preventive measures.  
- Blood‑glucose monitoring (fasting, post‑prandial, HbA1c) and lipid panels (total, LDL, HDL, triglycerides) help evaluate diabetes and heart‑disease risk, but single values are insufficient for diagnosis.  
- CBC, kidney (creatinine, eGFR) and liver (ALT, AST, bilirubin) tests provide insight into organ function; interpretation requires the full clinical picture.  
- Accurate, secure record‑keeping and prompt professional review of abnormal or symptomatic findings are essential for safe, effective care.


# Task 6: Comparision with Prompt-Based Summary
## Compare:
## Prompt-only summary
## Stuff chain summary
## Dicuss difference in output quality and limitations

# Feature | Prompt-only Summary | Stuff Chain Summary 

## Approach | Custom prompt + LLM | Document-combining summarization 
## Input     |  Extracted text          | Complete document content 
## Control   | High — prompt can be customized | Moderate 
## Output | Depends heavily on prompt | Generally consistent for document summarization 
## Large documents | Can exceed context window | Also limited by context window 
## Complexity  | Simple             | More structured 
## Best for | Custom instructions | Small/medium documents 

# PART 3 -- Map-reduce summarization chain


# Task 7: Why Map-Reduce is Needed 
## Answer briefly
## 1. Why large documents need map -reduce summarization
## 2. How map and reduce steps work

## 1.Why do large documents need Map-Reduce summarization?
## Map-Reduce summarization is useful for large documents that cannot fit into an LLM's context window at once. In the Map step, the document is divided into smaller chunks and each chunk is summarized independently.

# 2. How map and reduce steps work
## In the Reduce step, all the generated summaries are combined and passed to the LLM to produce a final overall summary. This approach makes it possible to summarize documents much larger than the model's single-input capacity.

# Task 8: Implement Map -reduce summarization Chain.
## 1. Split the document into chunks using text splitters
## 2. Use load_summarize_chain()with: chain_type="map_reduce"
## 3. generate the summary

In [12]:
from langchain_classic.chains.summarize import load_summarize_chain

model=ChatGroq(model="openai/gpt-oss-120b")
loader=PyPDFLoader("medical_health_data_document.pdf")
data=loader.load()


## splitting document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(data)


In [13]:

# 4. Create Map-Reduce summarization chain
summary_chain = load_summarize_chain(
    model,
    chain_type="map_reduce"
)
# 5. Generate summary
result = summary_chain.invoke({
    "input_documents": split_docs
})

# 6. Print final summary
print("\nFinal Summary:")
print(result["output_text"])


c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))



Final Summary:
**Concise Summary**

- **Medical health data** includes vitals, labs, history, medications, and preventive‑care records. Clinicians use them to assess status, identify risks, track trends, and guide decisions. The information is educational, not a substitute for professional advice.

- **Key Vital‑Sign Ranges (adults)**
  - Temperature: 36.1 – 37.2 °C  
  - Heart rate: 60 – 100 bpm  
  - Respiratory rate: 12 – 20 /min  
  - Blood pressure: ≈120/80 mmHg (multiple readings recommended)  
  - SpO₂: 95 %–100 %

- **Blood Pressure**
  - Systolic/diastolic (mmHg); varies with stress, activity, meds, etc.  
  - Persistent elevation ↑ cardiovascular & kidney disease risk.  
  - Prevention: exercise, low‑sodium diet, weight control, stress management, no tobacco, medication adherence.

- **Blood Glucose & HbA1c**
  - Fasting/post‑prandial glucose assesses diabetes risk/control.  
  - HbA1c reflects average glucose over 2‑3 months; diagnosis requires multiple data points.

- **Li

# PART 4--- Refine Summarization chain

# Task 10: Understanding Refine Chain
## Answer briefly:
## 1. How refine summarization works
## 2. Difference between map_reduce and refine

## 1 How does Refine Summarization work?
## A Refine Chain creates an initial summary from the first chunk of a document. Then, each following chunk is given to the LLM along with the existing summary, and the LLM refines/updates the summary using the new information.

# Difference between map_reduce and refine

##  Feature | Map-Reduce | Refine 
## Process | Summarizes chunks independently | Processes chunks sequentially 
## First step | Creates summaries for all chunks | Creates initial summary 
## Next chunks | Summarized independently | Used to update existing summary 
## Final step | Combines partial summaries | Final summary is already refined 
##  Speed | Usually faster | Usually slower |
## API cost | Multiple LLM calls | Multiple sequential LLM calls |

# Task 11: Implement Refine Summarization Chain
##  1. use laod_summarizechain() with: chain_type="refine"
## 2. generate the refined summary iteratively

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(split_docs))

Number of chunks: 9


In [15]:
## create the inital summary
from langchain_core.prompts import PromptTemplate

initial_prompt = PromptTemplate.from_template("""
You are an expert medical document summarizer.

Create a clear and concise summary of the following document section.

Document section:
{text}

Initial Summary:
""")

In [21]:
## create the refined prompt
refine_prompt = PromptTemplate.from_template("""
You are an expert medical document summarizer.

We already have the following summary:

Existing Summary:
{existing_answer}

Now consider this new section of the document:

New Document Section:
{text}

Refine the existing summary by incorporating important information
from the new section.

Keep the summary clear, concise, and accurate.
Do not remove important information from the existing summary.

Refined Summary:
""")

In [22]:

# 4. Create Refine summarization chain
summary_chain = load_summarize_chain(
    model,
    chain_type="refine",
    question_prompt=initial_prompt,
    refine_prompt=refine_prompt
    
    
)

# 5. Generate refined summary iteratively
result = summary_chain.invoke({
    "input_documents": split_docs
})

# 6. Display final summary
print("\nRefined Summary:")
print(result["output_text"])



Refined Summary:
**Refined Summary – “Medical Health Data: An Overview of Common Health Indicators”**

---

### 1. Purpose & Scope  
- **Educational reference (3‑5 pages)** describing typical medical health indicators, basic interpretation, and best‑practice handling of the data.  
- **Not a substitute for professional diagnosis or medical advice.**  

---

### 2. What Is Medical Health Data?  
- Information gathered on a person’s physical condition, symptoms, measurements, laboratory results, medical history, medications, and preventive‑care activities.  
- Used by clinicians to assess health status, detect risks, track changes over time, and guide clinical decisions.  
- **Interpretation caveat:** Meaning of any result depends on the individual, measurement method, timing, and broader clinical context; a single abnormal value rarely establishes a diagnosis.

---

### 3. Key Vital‑Sign Measurements  

| Indicator | What it measures | Typical adult reference* |
|-----------|----------

# Task 12: Comparision of all summarization methods
## compare:
## Prompt-based
## stuff chain
## Map-reduce chain
## Refine chain

## Evaluate based on:
## Summary quality
## coherence
## suitablity for long documents

# Method | Summary Quality | Coherence | Suitability for Long Documents |

## Prompt-based | Good, depends heavily on prompt quality | Good for smaller documents | Low 
## Stuff Chain | Very good for small/medium documents | Very good because the whole document is considered together |Low 
## Map-Reduce | Good to very good | Good, but some connections between sections may be lost | High 
## Refine Chain** | Very good | Excellent because the summary is continuously updated | High 

# PART -5 Mini Project: Documnet Summarizer
# Task 13: Build a summarization Function
## Create a reusable function:
## def summarize_document(text,method="map_reduce"):
## returns summary
## allow switching between:
## prompt
## stuff map_reduce
## refine

In [ ]:

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain

load_dotenv()

# Initialize LLM
model = ChatGroq(model="openai/gpt-oss-120b")

# Load PDF
loader = PyPDFLoader("medical_health_data_document.pdf")
data = loader.load()


def summarize_document(text, method="map_reduce"):
    """
    Summarize text or a list of LangChain Documents.

    Supported methods:
    prompt, stuff, map_reduce, refine

    Returns:
        str: Generated summary
    """

    # Accept either a string or a list of Documents
    if isinstance(text, str):
        documents = [Document(page_content=text)]
    else:
        documents = text

    if not documents:
        return ""

    # Split long documents into manageable chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(documents)

    # PROMPT METHOD
    if method == "prompt":
        prompt = PromptTemplate.from_template("""
        Summarize the following document in 5-6 lines.
        Include the main ideas and important information.

        Document:
        {text}

        Summary:
        """)

        chain = load_summarize_chain(
            model,
            chain_type="stuff",
            prompt=prompt
        )

    # STUFF METHOD
    elif method == "stuff":
        prompt = PromptTemplate.from_template("""
        Summarize the document clearly and concisely.
        Include the key facts and findings.

        Document:
        {text}

        Summary:
        """)

        chain = load_summarize_chain(
            model,
            chain_type="stuff",
            prompt=prompt
        )

    # MAP-REDUCE METHOD
    elif method == "map_reduce":
        map_prompt = PromptTemplate.from_template("""
        Summarize this document section, preserving key information.

        Section:
        {text}

        Summary:
        """)

        combine_prompt = PromptTemplate.from_template("""
        Combine these partial summaries into one coherent summary.
        Remove repetition and preserve important information.

        Partial summaries:
        {text}

        Final summary:
        """)

        chain = load_summarize_chain(
            model,
            chain_type="map_reduce",
            map_prompt=map_prompt,
            combine_prompt=combine_prompt
        )

    # REFINE METHOD
    elif method == "refine":
        initial_prompt = PromptTemplate.from_template("""
        Create an initial summary of this document section.

        Section:
        {text}

        Summary:
        """)

        refine_prompt = PromptTemplate.from_template("""
        Existing summary:
        {existing_answer}

        New document section:
        {text}

        Improve the existing summary by adding important
        information from the new section. Avoid repetition.

        Updated summary:
        """)

        chain = load_summarize_chain(
            model,
            chain_type="refine",
            question_prompt=initial_prompt,
            refine_prompt=refine_prompt
        )

    else:
        raise ValueError(
            "Invalid method. Choose: prompt, stuff, "
            "map_reduce, or refine."
        )

    # Generate and return summary
    response = chain.invoke({"input_documents": chunks})
    return response["output_text"]


# Example: Generate a summary from the PDF
summary = summarize_document(data, method="map_reduce")
print(summary)


c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


**Medical Health Data – Integrated Overview**

**Purpose & Scope**  
This educational guide (3‑5 pages) introduces the most common medical health indicators, basic preventive‑care concepts, and simple ways to interpret health data. It is intended for learning only and does not replace professional medical diagnosis or advice.

**What Constitutes Medical Health Data?**  
Medical health data includes a person’s vital signs, anthropometric measurements, laboratory test results, medical history, medications, and preventive‑care activities. Clinicians use these data to assess current health, identify risk factors, track changes over time, and guide clinical decisions.

---

### 1. Core Vital Signs & Anthropometry  
| Indicator | Typical Adult Reference Range* | Clinical Meaning |
|-----------|------------------------------|------------------|
| **Body temperature** | 36.1 – 37.2 °C (core) | Infection or metabolic status |
| **Heart rate (pulse)** | 60 – 100 bpm (resting) | Cardiac function,

# Task 14: Observations & Insights
## write short answers:
## 1. Best summarization strategy for very long documents
## 2. Trade- offs between speed and quality
## 3. Real-world use cases of eacxh method

# 1. Best summarization strategy for very long documents
## For very large documents, Map-Reduce is usually a practical choice because it handles chunks independently and can scale better.

## 2. Trade offs 
## Method | Speed | Quality | Main Trade-off 

## Prompt-based |  Fast | Good | Simple but less structured 
## Stuff | Fast | Very Good | Limited by context window 
## Map-Reduce | Medium | Very Good | More LLM calls but handles large documents 
## Refine | Slow | Very Good | Sequential processing gives better continuity 

# Real world use cases of each
# Prompt-based
## . Summarizing emails
## . Short articles
## . Meeting notes
## . Small text documents

# Stuff Chain
## . 3–10 page reports
## . Research papers of manageable size
## .Medical reports
## .Short business documents

# Map-Reduce
## . Large research papers
## . Books and lengthy reports
## .Legal documents
## . Large collections of documents
## .News archives

# Refine Chain
## .Long reports where section-to-section continuity is important
## .Research documents
## .Technical documentation
## .Detailed medical or legal documents
## .Sequential reports